In [2]:
from pathlib import Path
import polars as pl
import numpy as np

ROOT = Path("/home/ubuntu/avito/competition")

print("Датасет существует:", ROOT.exists())

for split in ["train", "val", "test"]:
    folder = ROOT / split / "impressions"
    files = sorted(folder.glob("*.parquet"))

    print(f"{split}: {len(files)} parquet files")

    embedding = (
        ROOT / split / "item_embeddings" / "data.memmap"
    )
    print("  embeddings:", embedding.exists())


Датасет существует: True
train: 512 parquet files
  embeddings: True
val: 64 parquet files
  embeddings: True
test: 64 parquet files
  embeddings: True


In [4]:
train_files = sorted(
    (ROOT / "train" / "impressions").glob("*.parquet")
)

assert train_files, "Train Parquet не найдены"

cols = [
    "feed_id",
    "user_id",
    "item_id",
    "category_id",
    "target",
    "history__item_id",
    "history__category_id",
    "history__event_type",
    "history__event_source",
    "history__item_id__embedding_idx",
]

feeds = (
    pl.scan_parquet(str(train_files[0]))
    .select(cols)
    .head(3)
    .collect()
)

for feed in feeds.iter_rows(named=True):
    print("=" * 50)
    print("feed_id:", feed["feed_id"])
    print("user_id:", feed["user_id"])
    print("Кандидатов:", len(feed["item_id"]))
    print("История:", len(feed["history__item_id"]))
    print("Положительных:", sum(feed["target"]))
    print("Категории истории:",
          feed["history__category_id"][:15])
    print("Типы событий:",
          feed["history__event_type"][:15])


feed_id: 2304337
user_id: 56184
Кандидатов: 72
История: 256
Положительных: 1
Категории истории: [13, 13, 11, 13, 11, 11, 11, 13, 13, 13, 13, 13, 13, 13, 11]
Типы событий: [1, 1, 1, 1, 7, 4, 1, 8, 6, 5, 6, 1, 1, 1, 1]
feed_id: 1410373
user_id: 2340621
Кандидатов: 120
История: 256
Положительных: 3
Категории истории: [48, 48, 48, 10, 10, 10, 41, 11, 11, 11, 6, 26, 2, 6, 26]
Типы событий: [7, 4, 4, 1, 1, 1, 1, 2, 3, 1, 1, 1, 1, 1, 1]
feed_id: 83338
user_id: 1559895
Кандидатов: 273
История: 256
Положительных: 2
Категории истории: [5, 6, 5, 11, 6, 40, 5, 5, 5, 6, 6, 6, 6, 32, 5]
Типы событий: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
feed_id: 1633422
user_id: 1869191
Кандидатов: 220
История: 12
Положительных: 1
Категории истории: [2, 2, 16, 16, 16, 16, 16, 7, 7, 7, 7, 7]
Типы событий: [1, 1, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4]
feed_id: 2566964
user_id: 1343129
Кандидатов: 19
История: 162
Положительных: 1
Категории истории: [32, 6, 13, 13, 32, 32, 32, 32, 32, 6, 32, 32, 32, 32, 32]
Типы событий

In [5]:
feed = feeds.row(0, named=True)

candidate_cats = np.array(feed["category_id"])
history_cats = np.array(feed["history__category_id"])

hist_cat_count = np.array([
    np.sum(history_cats == cat)
    for cat in candidate_cats
])

result = pl.DataFrame({
    "item_id": feed["item_id"],
    "category_id": feed["category_id"],
    "hist_cat_count": hist_cat_count,
    "target": feed["target"],
})

print(result.head(15))


shape: (15, 4)
┌──────────┬─────────────┬────────────────┬────────┐
│ item_id  ┆ category_id ┆ hist_cat_count ┆ target │
│ ---      ┆ ---         ┆ ---            ┆ ---    │
│ i64      ┆ i64         ┆ i64            ┆ i64    │
╞══════════╪═════════════╪════════════════╪════════╡
│ 73328213 ┆ 3           ┆ 60             ┆ 0      │
│ 47564870 ┆ 13          ┆ 68             ┆ 0      │
│ 92744465 ┆ 11          ┆ 40             ┆ 0      │
│ 69672495 ┆ 7           ┆ 7              ┆ 1      │
│ 92017650 ┆ 13          ┆ 68             ┆ 0      │
│ …        ┆ …           ┆ …              ┆ …      │
│ 50133873 ┆ 11          ┆ 40             ┆ 0      │
│ 93079334 ┆ 13          ┆ 68             ┆ 0      │
│ 9290071  ┆ 10          ┆ 21             ┆ 0      │
│ 80199910 ┆ 50          ┆ 5              ┆ 0      │
│ 33446981 ┆ 10          ┆ 21             ┆ 0      │
└──────────┴─────────────┴────────────────┴────────┘


In [6]:
result.shape


(72, 4)

In [7]:
sample = Path(
    "/home/ubuntu/avito/competition/sample_submission.parquet"
)

print("Файл существует:", sample.exists())

if sample.exists():
    df = pl.read_parquet(sample)
    print("Строк:", df.height)
    print(df.head())

Файл существует: True
Строк: 2904806
shape: (5, 3)
┌─────────┬───────────┬───────┐
│ feed_id ┆ item_id   ┆ score │
│ ---     ┆ ---       ┆ ---   │
│ i32     ┆ i32       ┆ f32   │
╞═════════╪═══════════╪═══════╡
│ 27586   ┆ 50639592  ┆ 0.0   │
│ 27586   ┆ 77707727  ┆ 0.0   │
│ 27586   ┆ 110441326 ┆ 0.0   │
│ 27586   ┆ 99506506  ┆ 0.0   │
│ 27586   ┆ 45124951  ┆ 0.0   │
└─────────┴───────────┴───────┘


In [8]:
df.shape

(2904806, 3)

In [9]:

# Проверяем размер полной таблицы
print("Полная таблица:", result.shape)

# Показываем только положительные объявления
positives = result.filter(pl.col("target") == 1)

print("Положительных объявлений:", positives.height)
print(positives)

# Статистика нового признака
print(result["hist_cat_count"].describe())

# Проверяем длины
assert result.height == len(feed["item_id"])
assert result["hist_cat_count"].min() >= 0
assert result["hist_cat_count"].max() <= len(
    feed["history__category_id"]
)

print("Проверки пройдены!")


Полная таблица: (72, 4)
Положительных объявлений: 1
shape: (1, 4)
┌──────────┬─────────────┬────────────────┬────────┐
│ item_id  ┆ category_id ┆ hist_cat_count ┆ target │
│ ---      ┆ ---         ┆ ---            ┆ ---    │
│ i64      ┆ i64         ┆ i64            ┆ i64    │
╞══════════╪═════════════╪════════════════╪════════╡
│ 69672495 ┆ 7           ┆ 7              ┆ 1      │
└──────────┴─────────────┴────────────────┴────────┘
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 72.0      │
│ null_count ┆ 0.0       │
│ mean       ┆ 41.152778 │
│ std        ┆ 24.41926  │
│ min        ┆ 0.0       │
│ 25%        ┆ 21.0      │
│ 50%        ┆ 40.0      │
│ 75%        ┆ 68.0      │
│ max        ┆ 68.0      │
└────────────┴───────────┘
Проверки пройдены!


In [10]:
feed = feeds.row(0, named=True)

targets = np.asarray(feed["target"])
items = np.asarray(feed["item_id"])

print("Форма вектора target:", targets.shape)
print("Первые 20 значений:", targets[:20])

positive_indices = np.where(targets == 1)[0]

print("Индексы положительных:", positive_indices)
print("ID положительных объявлений:", items[positive_indices])

assert len(targets) == len(items)


Форма вектора target: (72,)
Первые 20 значений: [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Индексы положительных: [3]
ID положительных объявлений: [69672495]
